# 02 - The data pipeline

Two DVC stages, declared in `dvc.yaml`:

```
data/raw/player_churn.csv
        │  features   (churn.features)   select → fill → one-hot → int label
        ▼
data/interim/features.parquet
        │  split      (churn.split)      stratified split → balance TRAIN only
        ▼
data/processed/{train,validation,test_x,test_y,baseline}.csv
```

Both stages are pure pandas: no AWS, no MLflow. That is what makes them
reproducible anywhere, which is what makes DVC's hashes meaningful.

### The one methodological fix

The original workshop oversampled the minority class in notebook 01, ingested the
duplicated rows into Feature Store, and split in notebook 02 — **in that order**.
So copies of the same player landed in both train and test, and the test fold was
also rebalanced to 50/50. The reported AUC measured partly memorisation, on a
distribution that does not exist.

Here: stratified split first, oversample the **train fold only**. Validation and
test keep the real 8.4% churn rate.

Expect a **lower and more honest** AUC than the workshop reported. That is the fix
working, not a regression.

In [1]:
import json
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
os.environ["CHURN_PROJECT_ROOT"] = str(PROJECT_ROOT)

# Put this kernel's venv bin/ first on PATH so shell commands (dvc, ...) resolve to the
# venv, however Jupyter was launched.
import sys
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"
for line in (PROJECT_ROOT / ".env").read_text().strip().splitlines():
    key, _, value = line.partition("=")
    os.environ[key] = value


def run(cmd, check=True):
    print("$", cmd)
    proc = subprocess.run(cmd, shell=True, cwd=PROJECT_ROOT, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print(proc.stderr.rstrip())
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")
    return proc

## 1. Read the pipeline definition

`deps` is what invalidates a stage. Note `src/churn/features.py` is listed: edit
the code and DVC knows the output is stale, same as if the data had changed.
`params` pulls specific `params.yaml` sections into the hash, so bumping
`split.random_state` reruns `split` but **not** `features`.

In [2]:
print((PROJECT_ROOT / "dvc.yaml").read_text())

# DVC owns DATA lineage only: raw -> features -> splits.
#
# Training/evaluation deliberately stay OUT of this graph. They run as remote
# SageMaker jobs and their lineage lives in MLflow, which already stores params,
# metrics and the model artifact. Making them DVC stages would churn dvc.lock on
# every experiment and give you two competing metric stores for the same run.
#
# The join between the two systems: every MLflow run is tagged with the md5 of
# the DVC outputs it consumed (see churn.dvc_meta), so any run can be traced
# back to an exact `dvc checkout`.

stages:
  features:
    cmd: python -m churn.features --input data/raw/player_churn.csv --output data/interim/features.parquet
    deps:
      - data/raw/player_churn.csv
      - src/churn/features.py
    params:
      - features
      - project.target_col
      - project.id_col
    outs:
      - data/interim/features.parquet

  split:
    cmd: python -m churn.split --input data/interim/features.parquet --outdir data/processe

In [3]:
run("dvc dag")

$ dvc dag
+-------------------------------+  
| data/raw/player_churn.csv.dvc |  
+-------------------------------+  
                *                  
                *                  
                *                  
          +----------+             
          | features |             
          +----------+             
                *                  
                *                  
                *                  
            +-------+              
            | split |              
            +-------+


CompletedProcess(args='dvc dag', returncode=0, stdout='+-------------------------------+  \n| data/raw/player_churn.csv.dvc |  \n+-------------------------------+  \n                *                  \n                *                  \n                *                  \n          +----------+             \n          | features |             \n          +----------+             \n                *                  \n                *                  \n                *                  \n            +-------+              \n            | split |              \n            +-------+              \n', stderr='')

## 2. Run it

`dvc repro` executes only stages whose inputs changed. First run does both.

In [4]:
run("dvc repro")

$ dvc repro
'data/raw/player_churn.csv.dvc' didn't change, skipping
Stage 'features' didn't change, skipping
Stage 'split' didn't change, skipping
Data and pipelines are up to date.


CompletedProcess(args='dvc repro', returncode=0, stdout="'data/raw/player_churn.csv.dvc' didn't change, skipping\nStage 'features' didn't change, skipping\nStage 'split' didn't change, skipping\nData and pipelines are up to date.\n", stderr='')

In [5]:
summary = json.loads((PROJECT_ROOT / "reports/split_summary.json").read_text())
print(json.dumps(summary, indent=2))

{
  "n_rows_total": 1640,
  "n_features": 28,
  "train_rows_before_balance": 1148,
  "train_rows": 2102,
  "validation_rows": 328,
  "test_rows": 164,
  "train_positive_rate": 0.5,
  "validation_positive_rate": 0.08231707317073171,
  "test_positive_rate": 0.08536585365853659,
  "balanced_train": true,
  "random_state": 1729
}


Read those numbers:

- `train_rows` > `train_rows_before_balance` -- the train fold was oversampled.
- `train_positive_rate` ≈ 0.50 -- balanced, as XGBoost wants.
- `test_positive_rate` ≈ 0.085 -- **not** balanced. The real churn rate, preserved.

If `test_positive_rate` ever comes back near 0.5, the leak is back.

In [6]:
import pandas as pd

train = pd.read_csv("data/processed/train.csv", header=None)
test_x = pd.read_csv("data/processed/test_x.csv", header=None)
print(f"train      {train.shape}  (label in column 0, no header)")
print(f"test_x     {test_x.shape}")
print(f"features   {train.shape[1] - 1}")
train.head(3)

train      (2102, 29)  (label in column 0, no header)
test_x     (164, 28)
features   28


,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
0,1,2,67536.554228,102780.163032,68041.096402,27511.542770,20285.540930,0.0,75310.148488,58929.985456,...,0,0,0,1,0,0,0,0,0,0
1,0,6,0.000000,7166.024739,56513.294268,63679.319008,56513.294268,0.0,63679.319008,0.000000,...,0,0,0,0,0,0,0,1,0,0
2,0,6,0.000000,7154.791579,80093.866233,848.657811,80093.866233,0.0,848.657811,0.000000,...,0,0,0,0,0,0,0,1,0,0


## 3. `dvc.lock` is the record

This is the file that makes an MLflow run traceable. Every output carries a
content hash; notebook 03 copies these onto the run as tags.

In [8]:
from src.churn.dvc_meta import describe, lineage_tags, output_hashes

print("outputs:")
for path, digest in output_hashes().items():
    print(f"  {path:35s} {digest}")

print("\nlineage tags that will go on every MLflow run:")
for key, value in sorted(lineage_tags().items()):
    print(f"  {key:30s} {value}")

print("\n" + describe())

outputs:
  data/interim/features.parquet       d0605019cf55f50aaeee3c993fe9847a
  data/processed                      5f68eff31363915f49ff8091544bd6e0.dir
  reports/split_summary.json          bf9ed4c83e081dac839a58512490eb6d

lineage tags that will go on every MLflow run:
  dvc.data.interim.features.parquet d0605019cf55f50aaeee3c993fe9847a
  dvc.data.processed             5f68eff31363915f49ff8091544bd6e0.dir
  dvc.reports.split_summary.json bf9ed4c83e081dac839a58512490eb6d
  dvc_pipeline_stages            features,split
  git_commit                     694fa803f80447ae330dfb1cbf5e9b9d8a9097d9-dirty

git 694fa803f804 / data.interim.features.parquet=d0605019, data.processed=5f68eff3, reports.split_summary.json=bf9ed4c8


## 4. Selective reruns

Change one param and watch DVC rerun the minimum. This is the behaviour you are
paying for: no "did I remember to re-run notebook 01?".

In [9]:
# Demonstration only -- reverted in the next cell.
import re

params_path = PROJECT_ROOT / "params.yaml"
original = params_path.read_text()
params_path.write_text(original.replace("random_state: 1729", "random_state: 99"))

run("dvc status")

$ dvc status
split:
	changed deps:
		params.yaml:
			modified:           split


CompletedProcess(args='dvc status', returncode=0, stdout='split:\n\tchanged deps:\n\t\tparams.yaml:\n\t\t\tmodified:           split\n', stderr='')

In [10]:
# `features` is unaffected (its params section did not change); only `split` is stale.
params_path.write_text(original)
run("dvc status")
print("reverted")

$ dvc status
Data and pipelines are up to date.
reverted


## 5. Push and commit

Same discipline as notebook 01: data first, then pointers.

In [11]:
run("dvc push")
run("git status --short -- churn-mlops", check=False)
print(
    """
Commit the lock file and the metrics summary:

  git add churn-mlops/dvc.lock churn-mlops/reports/split_summary.json
  git commit -m "churn-mlops: reproduce data pipeline"
"""
)

$ dvc push
7 files pushed
$ git status --short -- churn-mlops

Commit the lock file and the metrics summary:

  git add churn-mlops/dvc.lock churn-mlops/reports/split_summary.json
  git commit -m "churn-mlops: reproduce data pipeline"



---
## Next

**`03-train.ipynb`** -- a SageMaker training job, tracked in MLflow, pinned to
the DVC hashes you just produced.

### What changed vs. the original workshop
Old notebooks 01 + 02 (feature engineering, Feature Store ingest, Feature Store
query, split) collapse into `dvc repro`. The hand-written
`write_params`/`read_params` helpers that shuttled S3 paths through JSON files —
duplicated verbatim in four notebooks plus `model_evaluation/evaluation.py` — are
gone. `dvc.lock` does that job, and does it with hashes.